In [ ]:
# 

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import random
from scipy import stats
from localcider.sequenceParameters import SequenceParameters
import umap
MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}#from AAindex MUNV940103
import matplotlib
from adjustText import adjust_text
plt.rcParams["font.family"] = "Arial"
from sklearn import cluster, datasets, mixture
from scipy.stats import chi2_contingency
from adjustText import adjust_text


In [ ]:
TMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')
dSAM_df = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')

In [ ]:
dSAM_df[dSAM_df.AAseq.str.find('HSVIVMSS')!=-1]

In [ ]:
import sklearn.preprocessing
import sklearn.pipeline
import scipy.spatial
import numpy as np


def calculate_hull(
        X, 
        scale=1.1, 
        padding="scale", 
        n_interpolate=100, 
        interpolation="quadratic_periodic", 
        return_hull_points=False):
    """
    Calculates a "smooth" hull around given points in `X`.
    The different settings have different drawbacks but the given defaults work reasonably well.
    Parameters
    ----------
    X : np.ndarray
        2d-array with 2 columns and `n` rows
    scale : float, optional
        padding strength, by default 1.1
    padding : str, optional
        padding mode, by default "scale"
    n_interpolate : int, optional
        number of interpolation points, by default 100
    interpolation : str or callable(ix,iy,x), optional
        interpolation mode, by default "quadratic_periodic"

    Inspired by: https://stackoverflow.com/a/17557853/991496
    """
    
    if padding == "scale":

        # scaling based padding
        scaler = sklearn.pipeline.make_pipeline(
            sklearn.preprocessing.StandardScaler(with_std=False),
            sklearn.preprocessing.MinMaxScaler(feature_range=(-1,1)))
        points_scaled = scaler.fit_transform(X) * scale
        hull_scaled = scipy.spatial.ConvexHull(points_scaled, incremental=True)
        hull_points_scaled = points_scaled[hull_scaled.vertices]
        hull_points = scaler.inverse_transform(hull_points_scaled)
        hull_points = np.concatenate([hull_points, hull_points[:1]])
    
    elif padding == "extend" or isinstance(padding, (float, int)):
        # extension based padding
        # TODO: remove?
        if padding == "extend":
            add = (scale - 1) * np.max([
                X[:,0].max() - X[:,0].min(), 
                X[:,1].max() - X[:,1].min()])
        else:
            add = padding
        points_added = np.concatenate([
            X + [0,add], 
            X - [0,add], 
            X + [add, 0], 
            X - [add, 0]])
        hull = scipy.spatial.ConvexHull(points_added)
        hull_points = points_added[hull.vertices]
        hull_points = np.concatenate([hull_points, hull_points[:1]])
    else:
        raise ValueError(f"Unknown padding mode: {padding}")
    
    # number of interpolated points
    nt = np.linspace(0, 1, n_interpolate)
    
    x, y = hull_points[:,0], hull_points[:,1]
    
    # ensures the same spacing of points between all hull points
    t = np.zeros(x.shape)
    t[1:] = np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2)
    t = np.cumsum(t)
    t /= t[-1]

    # interpolation types
    if interpolation is None or interpolation == "linear":
        x2 = scipy.interpolate.interp1d(t, x, kind="linear")(nt)
        y2 = scipy.interpolate.interp1d(t, y, kind="linear")(nt)
    elif interpolation == "quadratic":
        x2 = scipy.interpolate.interp1d(t, x, kind="quadratic")(nt)
        y2 = scipy.interpolate.interp1d(t, y, kind="quadratic")(nt)

    elif interpolation == "quadratic_periodic":
        x2 = scipy.interpolate.splev(nt, scipy.interpolate.splrep(t, x, per=True, k=4))
        y2 = scipy.interpolate.splev(nt, scipy.interpolate.splrep(t, y, per=True, k=4))
    
    elif interpolation == "cubic":
        x2 = scipy.interpolate.CubicSpline(t, x, bc_type="periodic")(nt)
        y2 = scipy.interpolate.CubicSpline(t, y, bc_type="periodic")(nt)
    else:
        x2 = interpolation(t, x, nt)
        y2 = interpolation(t, y, nt)
    
    X_hull = np.concatenate([x2.reshape(-1,1), y2.reshape(-1,1)], axis=1)
    if return_hull_points:
        return X_hull, hull_points
    else:
        return X_hull


def draw_hull(
        X, 
        scale=1.1, 
        padding="scale", 
        n_interpolate=100, 
        interpolation="quadratic_periodic",
        plot_kwargs=None, 
        ax=None):
    """Uses `calculate_hull` to draw a hull around given points.

    Parameters
    ----------
    X : np.ndarray
        2d-array with 2 columns and `n` rows
    scale : float, optional
        padding strength, by default 1.1
    padding : str, optional
        padding mode, by default "scale"
    n_interpolate : int, optional
        number of interpolation points, by default 100
    interpolation : str or callable(ix,iy,x), optional
        interpolation mode, by default "quadratic_periodic"
    plot_kwargs : dict, optional
        `matplotlib.pyplot.plot` kwargs, by default None
    ax : `matplotlib.axes.Axes`, optional
        [description], by default None
    """

    if plot_kwargs is None:
        plot_kwargs = {}

    X_hull = calculate_hull(
        X, scale=scale, padding=padding, n_interpolate=n_interpolate, interpolation=interpolation)
    if ax is None:
        ax= plt.gca()
    plt.plot(X_hull[:,0], X_hull[:,1], **plot_kwargs)


def draw_rounded_hull(X, padding=0.1, line_kwargs=None, ax=None):
    """Plots a convex hull around points with rounded corners and a given padding.

    Parameters
    ----------
    X : np.array
        2d array with two columns and n rows
    padding : float, optional
        padding between hull and points, by default 0.1
    line_kwargs : dict, optional
        line kwargs (used for `matplotlib.pyplot.plot` and `matplotlib.patches.Arc`), by default None
    ax : matplotlib.axes.Axes, optional
        axes to plat on, by default None
    """

    default_line_kwargs = dict(
        color="black",
        linewidth=1
    )
    if line_kwargs is None:
        line_kwargs = default_line_kwargs
    else:
        line_kwargs = {**default_line_kwargs, **line_kwargs}

    if ax is None:
        ax = plt.gca()

    hull = scipy.spatial.ConvexHull(X)
    hull_points = X[hull.vertices]

    hull_points = np.concatenate([hull_points[[-1]], hull_points, hull_points[[0]]])

    diameter = padding * 2
    for i in range(1, hull_points.shape[0] - 1):

        # line
        
        # source: https://stackoverflow.com/a/1243676/991496
        
        norm_next = np.flip(hull_points[i] - hull_points[i + 1]) * [-1, 1]
        norm_next /= np.linalg.norm(norm_next)

        norm_prev = np.flip(hull_points[i - 1] - hull_points[i]) * [-1, 1]
        norm_prev /= np.linalg.norm(norm_prev)

        # plot line
        line = hull_points[i:i+2] + norm_next * diameter / 2
        ax.plot(line[:,0], line[:,1], **line_kwargs) 

        # arc

        angle_next = np.rad2deg(np.arccos(np.dot(norm_next, [1,0])))
        if norm_next[1] < 0:
            angle_next = 360 - angle_next

        angle_prev = np.rad2deg(np.arccos(np.dot(norm_prev, [1,0])))
        if norm_prev[1] < 0:
            angle_prev = 360 - angle_prev

        arc = patches.Arc(
            hull_points[i], 
            diameter, diameter,
            angle=0, fill=False, theta1=angle_prev, theta2=angle_next,
            **line_kwargs)

        ax.add_patch(arc)

In [ ]:
dSAM_df = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')

In [ ]:
dSAM_df = dSAM_df.drop_duplicates(subset='AAseq')

In [ ]:
dSAM_df.loc[:,['acc','AAseq','start_new','end_new','AA_length']].to_csv('STable_1.csv')

In [ ]:
dSAM_df#[dSAM_df.acc=='P12830']

In [ ]:
len(set(TMs.acc))
#len(set(dSAM_df.acc))
4923-945

In [ ]:
fig, ax = plt.subplots(figsize=(3*1.619,3))#
plt.pie([945,4923-945],labels=['$Yes$','$No$'],colors=['#C03830','#B4B5B4'],wedgeprops={'linewidth': 0})
#plt.setp(['$No$','$dSAM$'], size=8, weight="bold")
plt.title('$MP\;Harboring\;mdSAM?$')
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
plt.tight_layout()
plt.text(-.7,-.40,'$3978$\n$_{(81\%)}$')
plt.text(.2,.20,'$945$\n$_{(19\%)}$',color='w')
plt.savefig('mdSAMs_in_membrane_proteins_high_con.svg',transparent=True)

In [ ]:
l = []
for i in range(70):
    for j in range(70):
        l.append({'x':i,'y':j})

In [ ]:
mdSAMs = dSAM_df.drop_duplicates(subset='AAseq')
len(mdSAMs)

In [ ]:
df  = pd.DataFrame(l)
df = df.loc[:4886]

df['mdSAM'] = '-'
df.loc[df.sample(len(set(mdSAMs.acc))).index,'mdSAM']='+'
#df.loc[df[3941::].index,'mdSAM']='+'
fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='x',y='y',data=df,s=5,hue='mdSAM',hue_order=['+','-'],palette=['#C03830','#B4B5B4'])
plt.xlim([-1,70])
plt.ylim([-1,70])
plt.xticks([])
plt.yticks([])
plt.xlabel('')
plt.ylabel('')
plt.legend([],frameon=False)
plt.savefig('mdSAMs_in_membrane_proteins_high_con_dot_matrix_random.svg',transparent=True)

In [ ]:
df  = pd.DataFrame(l)
df = df.loc[:4886]

df['mdSAM'] = '-'
#df.loc[df.sample(len(set(mdSAMs.acc))).index,'mdSAM']='+'
df.loc[df[3941::].index,'mdSAM']='+'
fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='x',y='y',data=df,edgecolor=None,s=5,hue='mdSAM',hue_order=['+','-'],palette=['#C03830','#B4B5B4'])

#ax.axis('off')
plt.xlim([-1,70])
plt.ylim([-1,70])
plt.xticks([])
plt.yticks([])
plt.xlabel('')
plt.ylabel('')
plt.legend([],frameon=False)
plt.savefig('mdSAMs_in_membrane_proteins_high_con_dot_matrix.svg',transparent=True)

In [ ]:

# create data
size_of_groups=[945,39]

# Create a pieplot
plt.pie(size_of_groups)

# add a circle at the center to transform it in a donut chart
my_circle=plt.Circle( (0,0), 0.57, color='white')
p=plt.gcf()
p.gca().add_artist(my_circle)


In [ ]:
l = []
for i in range(7,13):
    l.append({'Length':'%d'%i,'Count':len(dSAM_df[dSAM_df.AA_length==i])})
l.append({'Length':'≥13','Count':len(dSAM_df[dSAM_df.AA_length>=13])})
print(l)
fig, ax = plt.subplots(figsize=(2.5,2.5))#
sns.barplot(x='Length',y='Count',data=pd.DataFrame(l),color='#939598',width=.95,lw=1)
ax.grid()
plt.xlabel('Length (Amino acids)',fontsize=12)
plt.savefig('mdSAMs_length_distribution.svg',bbox_inches='tight',transparent=True)

In [ ]:
for i in dSAM_df[(dSAM_df.ENST.isin(AF2[AF2.acc.isin(set(TMs.acc))].enst))].index:
    #print(dSAM_df.loc[i,:])
    acc = AF2[AF2.enst==dSAM_df.loc[i,'ENST']].acc.values[0]
    dSAM_df.loc[i,'acc']=acc 

In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
umap_df = dSAM_df#.drop_duplicates(subset=['clustering_label'],keep='first')
umap_df = umap_df.reset_index()
umap_dff = pd.DataFrame()
for i in umap_df.index:
    #umap_dff.loc[i,'length'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_length()
    umap_dff.loc[i,'beta'] = np.mean([MUNV940103[i] for i in umap_df.loc[i,'AAseq']])
    umap_dff.loc[i,'hydropathy'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_uversky_hydropathy()
    umap_dff.loc[i,'mean_net_charge'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_mean_net_charge()
    umap_dff.loc[i,'Aromatic'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F']
    umap_dff.loc[i,'Posetive'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K']
    umap_dff.loc[i,'Negitive'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E']
    umap_dff.loc[i,'Hydro'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G']
    umap_dff.loc[i,'Polar'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C']
    


In [ ]:
sns.clustermap(pd.DataFrame(StandardScaler().fit_transform(umap_dff)))

In [ ]:
reducer = umap.UMAP(random_state=120,n_neighbors=5)
embedding = reducer.fit_transform(StandardScaler().fit_transform(umap_dff))
# for i in umap_df.index:
#     umap_dff.loc[i,'acc'] = umap_df.loc[i,'acc']
# #umap_dff.to_pickle('dSAMs_')

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
principalComponents = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=30,random_state=2).fit_transform(StandardScaler().fit_transform(umap_dff))

principalDf = pd.DataFrame(data = principalComponents
             , columns = ['x', 'y'])
UMAP_plot = pd.DataFrame()
UMAP_plot['x'] =principalDf.loc[:, 'x'] # #tsne_features[:,0] #
UMAP_plot['y'] =principalDf.loc[:, 'y'] # #tsne_features[:,0] #
for i in UMAP_plot.index:
    UMAP_plot.loc[i,'acc']=umap_df.loc[i,'acc']
    UMAP_plot.loc[i,'AAseq']=umap_df.loc[i,'AAseq']
    UMAP_plot.loc[i,'ENST']=umap_df.loc[i,'ENST']
    UMAP_plot.loc[i,'Agg_val'] = len(umap_df.loc[i,'AAseq'])
    UMAP_plot.loc[i,'delta'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_delta()
    UMAP_plot.loc[i,'hydropathy'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_uversky_hydropathy()
    UMAP_plot.loc[i,'fraction_disorder_promoting'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_fraction_disorder_promoting()
    UMAP_plot.loc[i,'length'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_length()
    UMAP_plot.loc[i,'FCR'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_FCR()
    UMAP_plot.loc[i,'NCPR'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_NCPR()
    UMAP_plot.loc[i,'Omega'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_Omega()
    UMAP_plot.loc[i,'mean_net_charge'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_mean_net_charge()
    UMAP_plot.loc[i,'Ala'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']
    UMAP_plot.loc[i,'Cys'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C']
    UMAP_plot.loc[i,'Asp'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']
    UMAP_plot.loc[i,'Glu'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E']
    UMAP_plot.loc[i,'Phe'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F']
    UMAP_plot.loc[i,'Gly'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G']
    UMAP_plot.loc[i,'His'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']
    UMAP_plot.loc[i,'Ile'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']
    UMAP_plot.loc[i,'Lys'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K']
    UMAP_plot.loc[i,'Leu'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']
    UMAP_plot.loc[i,'Met'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']
    UMAP_plot.loc[i,'Asn'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']
    UMAP_plot.loc[i,'Pro'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['P']
    UMAP_plot.loc[i,'Gln'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']
    UMAP_plot.loc[i,'Arg'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']
    UMAP_plot.loc[i,'Ser'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']
    UMAP_plot.loc[i,'Thr'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']
    UMAP_plot.loc[i,'Val'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']
    UMAP_plot.loc[i,'Trp'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']
    UMAP_plot.loc[i,'Tyr'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']
    UMAP_plot.loc[i,'Aromatic'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F'])
    UMAP_plot.loc[i,'Posetive'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K'])
    UMAP_plot.loc[i,'Negitive'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E'])
    UMAP_plot.loc[i,'Hydro'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G'])
    UMAP_plot.loc[i,'Polar'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C'])
    UMAP_plot.loc[i,'beta'] = np.sum([MUNV940103[i] for i in umap_df.loc[i,'AAseq']])
df_membrane  = UMAP_plot[~pd.isna(UMAP_plot.acc)]#
df_membrane = df_membrane.drop_duplicates(subset='AAseq')

In [ ]:
uniport = pd.read_csv('/Users/aelazar/notebooks_for_paper/idmapping_reviewed_true_2024_02_05.tsv',delimiter='\t')
d = {}
for i in uniport.index:    
    d['%s'%uniport.loc[i,'Entry']] = uniport.loc[i,'Gene Names'].split(' ')[0]
df_membrane['gene_name'] = df_membrane['acc'].map(d)
df_membrane    

In [ ]:
df_membrane= df_membrane.append({'x':100,'y':100,'Aromatic':100,'Posetive':100,'Negitive':100,'Hydro':100,'Polar':100},ignore_index=True)

In [ ]:
df_membrane= pd.concat([df_membrane,pd.DataFrame([{'x':100,'y':100,'Aromatic':100,'Posetive':100,'Negitive':100,'Hydro':100,'Polar':100}])],ignore_index=True)



In [ ]:
import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=0, vmax=100)

In [ ]:
fig = plt.figure(figsize = (6,9),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.1})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
l =['Negitive','Posetive','Aromatic','Hydro','Polar','beta']
ll = ['Negative(DE)','Positive(RKH)','Aromatic(YWF)',
      'Hydrophobic(GAVILM)','Polar(TSNQC)',r'$\sum\Delta G_{\beta} $']
# ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
#       '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
#ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
ll_colors  = [matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#24406A"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#832621"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#EC7E09"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#62415B"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#2E6060"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#B9A84B","#000832"])#B4B5B4
              ,'RdGy_r','RdGy_r','RdGy_r','RdGy_r','RdGy_r']

for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[i[0]],palette=ll_colors[i[0]],
                    ax=axs.flat[i[0]],linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False)
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title(ll[i[0]],fontsize=16)
    axs.flat[i[0]].legend([],frameon=False)
    axs.flat[i[0]].locator_params(axis='y', nbins=4)
    axs.flat[i[0]].locator_params(axis='x', nbins=4)
    axs.flat[i[0]].set_xlim([-60,55])
    axs.flat[i[0]].set_ylim([-60,55])
    #axs.flat[i[0]].axis('off')
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    #axs.flat[i[0]].set_xticklabels(['','', ''])
    #axs.flat[i[0]].set_yticklabels(['','', ''])
    sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]],norm=norm)
    #sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    cbar = fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.4,aspect=10)
    cbar.ax.locator_params(nbins=4)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    #plt.yticks([0,7.5,15],('', '', ''))
    axs.flat[i[0]].set_xlabel('t-SNE')
    axs.flat[i[0]].set_ylabel('t-SNE')
    axs.flat[i[0]].axis('off')
plt.tight_layout()
#plt.savefig('all_UMAP_madan_colors_paper_not_rasterized_v3.svg',transparent=True)
plt.savefig('all_tsne_colors_paper_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
fig = plt.figure(figsize = (5.5,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.0})

sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[1],palette=ll_colors[1]
                 ,linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False,ax=axs[0])
left, bottom, width, height = (-15, 20, 25, 25)
rect = plt.Rectangle((left, bottom), width, height,
                     facecolor="black", alpha=1)
axs[0].add_patch(rect)
axs[0].legend([],frameon=False)
axs[0].set_xlim([-60,55])
axs[0].set_ylim([-60,55])
axs[0].axis('off')
inds = [1032,1215,1223,1413]
# fig, ax = plt.subplots(figsize=(3, 3))
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[1],palette=ll_colors[1]
                 ,linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.index.isin(inds)],color='white'
                 ,linewidth=1,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1],zorder=0)
axs[1].set_xlim([-15,-15+25])
axs[1].set_ylim([20,20+25])
axs[1].axis('off')
axs[1].legend([],frameon=False)
texts = []
for i in inds:
    texts.append(axs[1].text(df_membrane.loc[i,'x'],df_membrane.loc[i,'y'],
            '%s'%(df_membrane.loc[i,'AAseq']),
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs[1],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('tnse_RK_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig = plt.figure(figsize = (5.5,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.0})

sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[1],palette=ll_colors[1]
                 ,linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False,ax=axs[0])
left, bottom, width, height = (-15, 20, 25, 25)
rect = plt.Rectangle((left, bottom), width, height,
                     facecolor="black", alpha=1)
axs[0].add_patch(rect)
axs[0].legend([],frameon=False)
axs[0].set_xlim([-60,55])
axs[0].set_ylim([-60,55])
axs[0].axis('off')
inds = [1032,1215,1223,1413]
# fig, ax = plt.subplots(figsize=(3, 3))
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[1],palette=ll_colors[1]
                 ,linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.index.isin(inds)],color='white'
                 ,linewidth=1,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1],zorder=0)
axs[1].set_xlim([-15,-15+25])
axs[1].set_ylim([20,20+25])
axs[1].axis('off')
axs[1].legend([],frameon=False)
texts = []
for i in inds:
    texts.append(axs[1].text(df_membrane.loc[i,'x'],df_membrane.loc[i,'y'],
            '%s'%(df_membrane.loc[i,'AAseq']),
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs[1],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('tnse_RK_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane.loc[inds,['AAseq','acc']]

In [ ]:
fig = plt.figure(figsize = (5.5,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.0})

sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[0],palette=ll_colors[0]
                 ,linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False,ax=axs[0])
left, bottom, width, height = (-58, 3, 25, 25)
rect = plt.Rectangle((left, bottom), width, height,
                     facecolor="black", alpha=.1)
axs[0].add_patch(rect)
axs[0].legend([],frameon=False)
axs[0].set_xlim([-60,55])
axs[0].set_ylim([-60,55])
axs[0].axis('off')
inds = [277,910,392,12]
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[0],palette=ll_colors[0]
                 ,linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.index.isin(inds)],color='white'
                 ,linewidth=1,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1],zorder=0)
axs[1].set_xlim([-58,-58+25])
axs[1].set_ylim([3,3+25])
axs[1].axis('off')
axs[1].legend([],frameon=False)
texts = []
for i in inds:
    texts.append(axs[1].text(df_membrane.loc[i,'x'],df_membrane.loc[i,'y'],
            '%s'%(df_membrane.loc[i,'AAseq']),
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs[1],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('tnse_DE_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane.loc[inds,['AAseq','acc']]

In [ ]:
fig = plt.figure(figsize = (5.5,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.0})

sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[2],palette=ll_colors[2]
                 ,linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False,ax=axs[0])
left, bottom, width, height = (-5, -20, 25, 25)
rect = plt.Rectangle((left, bottom), width, height,
                     facecolor="black", alpha=.1)
axs[0].add_patch(rect)
axs[0].legend([],frameon=False)
axs[0].set_xlim([-60,55])
axs[0].set_ylim([-60,55])
axs[0].axis('off')
inds = [87,707,1340,704]
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[2],palette=ll_colors[2]
                 ,linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.index.isin(inds)],color='white'
                 ,linewidth=1,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1],zorder=0)
axs[1].set_xlim([-5,-5+25])
axs[1].set_ylim([-20,-20+25])
axs[1].axis('off')
axs[1].legend([],frameon=False)
texts = []
for i in inds:
    texts.append(axs[1].text(df_membrane.loc[i,'x'],df_membrane.loc[i,'y'],
            '%s'%(df_membrane.loc[i,'AAseq']),
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs[1],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('tnse_YWF_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig = plt.figure(figsize = (6,9),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.1})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,edgecolor=None,color='#B4B5B4',s=10,ax=axs.flat[0])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.AAseq.isin(['LLCWYRT'])],edgecolor=None,color='r',s=10,ax=axs.flat[0])

axs.flat[0].set_xlim([-60,55])
axs.flat[0].set_ylim([-60,55])
axs.flat[0].axis('off')
plt.savefig('tnse_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig = plt.figure(figsize = (6,9),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.1})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
l =['Negitive','Posetive','Aromatic','Hydro','Polar','beta']
ll = ['Negative(DE)','Positive(RKH)','Aromatic(YWF)',
      'Hydrophobic(GAVILM)','Polar(TSNQC)',r'$\sum\Delta G_{\beta} $']
# ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
#       '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
#ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
ll_colors  = [matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#24406A"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#832621"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#EC7E09"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#62415B"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#DEE0E3","#2E6060"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#B9A84B","#000832"])#B4B5B4
              ,'RdGy_r','RdGy_r','RdGy_r','RdGy_r','RdGy_r']

for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[i[0]],palette=ll_colors[i[0]],
                    ax=axs.flat[i[0]],linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False)
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title(ll[i[0]],fontsize=16)
    axs.flat[i[0]].legend([],frameon=False)
    axs.flat[i[0]].locator_params(axis='y', nbins=4)
    axs.flat[i[0]].locator_params(axis='x', nbins=4)
    axs.flat[i[0]].set_xlim([-60,55])
    axs.flat[i[0]].set_ylim([-60,55])
    #axs.flat[i[0]].axis('off')
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    #axs.flat[i[0]].set_xticklabels(['','', ''])
    #axs.flat[i[0]].set_yticklabels(['','', ''])
    sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]],norm=norm)
    #sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    cbar = fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.4,aspect=10)
    cbar.ax.locator_params(nbins=4)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    #plt.yticks([0,7.5,15],('', '', ''))
    axs.flat[i[0]].set_xlabel('t-SNE')
    axs.flat[i[0]].set_ylabel('t-SNE')
    axs.flat[i[0]].axis('off')
plt.tight_layout()
#plt.savefig('all_UMAP_madan_colors_paper_not_rasterized_v3.svg',transparent=True)
plt.savefig('all_tsne_colors_paper_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig = plt.figure(figsize = (5.5,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.0})

sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[3],palette=ll_colors[3]
                 ,linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False,ax=axs[0])
left, bottom, width, height = (20, -20, 25, 25)
rect = plt.Rectangle((left, bottom), width, height,
                     facecolor="black", alpha=.1)
axs[0].add_patch(rect)
axs[0].legend([],frameon=False)
axs[0].set_xlim([-60,55])
axs[0].set_ylim([-60,55])
axs[0].axis('off')
inds = [533,1111,1313,1241]
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[3],palette=ll_colors[3]
                 ,linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.index.isin(inds)],color='white'
                 ,linewidth=1,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1],zorder=0)
axs[1].set_xlim([20,20+25])
axs[1].set_ylim([-20,-20+25])
axs[1].axis('off')
axs[1].legend([],frameon=False)
texts = []
for i in inds:
    texts.append(axs[1].text(df_membrane.loc[i,'x'],df_membrane.loc[i,'y'],
            '%s'%(df_membrane.loc[i,'AAseq']),
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs[1],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('tnse_Hydro_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig = plt.figure(figsize = (5.5,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.1,'wspace':.0})

sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[4],palette=ll_colors[4]
                 ,linewidth=0,edgecolor='k',alpha=1,s=10, rasterized=False,ax=axs[0])
left, bottom, width, height = (0, -25, 25, 25)
rect = plt.Rectangle((left, bottom), width, height,
                     facecolor="black", alpha=.1)
axs[0].add_patch(rect)
axs[0].legend([],frameon=False)
axs[0].set_xlim([-60,55])
axs[0].set_ylim([-60,55])
axs[0].axis('off')
inds = [73,73,672,70]
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[4],palette=ll_colors[4]
                 ,linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1])
sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.index.isin(inds)],color='white'
                 ,linewidth=1,edgecolor='k',alpha=1,s=30, rasterized=False,ax=axs[1],zorder=0)
axs[1].set_xlim([-0,0+25])
axs[1].set_ylim([-25,-25+25])
axs[1].axis('off')
axs[1].legend([],frameon=False)
texts = []
for i in inds:
    texts.append(axs[1].text(df_membrane.loc[i,'x'],df_membrane.loc[i,'y'],
            '%s'%(df_membrane.loc[i,'AAseq']),
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs[1],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('tnse_PLOAR_test.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane = df_membrane.iloc[:-1,:]

In [ ]:
df_membrane = pd.read_pickle('HTS/umap.pkl')
# for i in df_membrane.index:
#     df_membrane.loc[i,'gene_name'] = hTMs_uniprot.loc[hTMs_uniprot[hTMs_uniprot.Entry==df_membrane.loc[i,'acc']].index]['Gene Names'].values[0].split(' ')[0]
gmm = mixture.GaussianMixture(n_components=3, covariance_type="full",random_state=42)
gmm.fit(df_membrane.loc[:,['x','y']])

for i in enumerate(gmm.predict(df_membrane.loc[:,['x','y']])):
    df_membrane.at[df_membrane.index[i[0]],'gmm_dSAMs'] = i[1]

In [ ]:
df_membrane

In [ ]:
for jj in [157]:#range(1,10001):
    for n in [10]:#range(10,15):
    #print(n)
        gmm = mixture.GaussianMixture(n_components=n, covariance_type="diag",random_state=jj)
        gmm.fit(df_membrane.loc[:,['x','y']])

        for i in enumerate(gmm.predict(df_membrane.loc[:,['x','y']])):
            df_membrane.at[df_membrane.index[i[0]],'gmm_dSAMs'] = i[1]

In [ ]:
KeyWord_df = pd.read_csv('/Users/aelazar/Downloads/keywords_all_2023_10_27.tsv',delimiter='\t')

In [ ]:
high_mdSAM_KeyWord= []
for key in set(KeyWord_df[KeyWord_df.Category.isin(['Disease','Molecular function','Biological process'])]['Keyword ID']):
    if len(df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find(key)!=-1].Entry.values)])>=10:
        high_mdSAM_KeyWord.append(key)
        #break
#for i in KeyWord_df[KeyWord_df.Name=='Tyrosine-protein kinase']['Keyword ID'].values[0]

In [ ]:
from mycolorpy import colorlist as mcp
color1=mcp.gen_color(cmap="tab10",n=10)
color1 = ['#414042','#96615D','#697793']

In [ ]:

df_membrane['gmm_color'] = df_membrane.gmm_dSAMs.map(dict(zip(list(range(3)), color1)))

In [ ]:
sns.scatterplot(x='x',y='y',data=df_membrane,hue='gmm_dSAMs')

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))#
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=20,color='#DEE0E3')
plt.legend([],frameon=False)
ax.axis('off')
for i,j in zip(list(range(3)), color1):#,'#E73490'['#674CAC','#BB1919','#3C4776']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1.0,plot_kwargs={'color':j,'lw':2})
# ax.set_xlim([-60,55])
# ax.set_ylim([-60,55])
plt.savefig('tnse_GMM_10.svg',bbox_inches='tight',transparent=True)

In [ ]:
def hypergeom(keyword,cluster):
    total_runners =  len(df_membrane) # No. of hTMs Nodes in the Graph 
    number_team_A = len(df_membrane[df_membrane.gmm_dSAMs==cluster]) # number mdSAM nodes in the Graph  
    number_winners = len(df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find(keyword)!=-1].Entry.values)])## No. of PPIs i.e Edges 
    number_win_from_A = len(df_membrane[(df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find(keyword)!=-1].Entry.values))&(df_membrane.gmm_dSAMs==cluster)]) ## PPIs that are mdSAM
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 

In [ ]:
l  =[]
for key in high_mdSAM_KeyWord:
    for cluster in range(3):
        o,e,p = hypergeom(key,cluster)
        l.append({'cluster':cluster,'key':key,'obs':o,'exp':e,'pval':p,'keyword_ID':KeyWord_df[KeyWord_df['Keyword ID']==key].Name.values[0]})
        

In [ ]:
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
enrich = pd.DataFrame(l)
enrich['fdr'] = fdr(enrich.pval)
enrich['log_fdr'] = -1*np.log10(enrich['fdr'])
enrich['obs-exp']= enrich['obs']-enrich['exp']
from adjustText import adjust_text

In [ ]:
for j in range(3):
    fig, ax = plt.subplots(figsize=(3,3)) 
    enrich_sub = enrich[enrich.cluster==j]
    sns.scatterplot(x='obs-exp',y='log_fdr',data=enrich_sub,s=50,color=color1[j]
                    ,edgecolor=None)
    texts = []
    for i in enrich_sub[enrich_sub.log_fdr>1.25].index:
        texts.append(plt.text(enrich_sub.loc[i,'obs-exp'],enrich_sub.loc[i,'log_fdr'],
                '%s'%enrich_sub.loc[i,'keyword_ID'],
                              fontsize=10))
    plt.ylim([0,6.5])
    plt.xlim([-15,15])
    ax.grid()
    plt.locator_params(nbins=4)
    plt.ylabel('')
    plt.xlabel('')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
    plt.savefig('obs_exp_Keywords_gmm_cluster_%d.svg'%j,bbox_inches='tight',transparent=True)
    

In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 
sns.scatterplot(x='obs-exp',y='log_fdr',data=enrich,s=50
                ,hue='cluster',hue_order=[0,1,2],palette=color1,edgecolor=None)
ax.grid()
# plt.ylim([0,7.9])
# plt.xlim([-20,20])
texts = []
for i in enrich[enrich.log_fdr>1.25].index:
    texts.append(plt.text(enrich.loc[i,'obs-exp'],enrich.loc[i,'log_fdr'],
            '%s'%enrich.loc[i,'keyword_ID'],
                          fontsize=12))
texts.append(plt.text(enrich.loc[15,'obs-exp'],enrich.loc[15,'log_fdr'],
        '%s'%enrich.loc[15,'keyword_ID'],
                      fontsize=12))
adjust_text(texts, autoalign='xy',
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)#,force_text=(-.2, .1),force_points=(-.1, 0.1), only_move={'points':'xy', 'text':'xy'},
plt.locator_params(axis='both',nbins=4)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
# plt.legend([])
# sm = plt.cm.ScalarMappable(cmap='tab10')
# sm.set_array([])
# #ax.get_legend().remove()
# ax.figure.colorbar(sm,fraction=0.03,shrink=.4, pad=0.01)
plt.savefig('obs_exp_Keywords_gmm.svg',bbox_inches='tight',transparent=True) 

In [ ]:
enrich[enrich.keyword_ID=='Cell adhesion']

In [ ]:
sns.barplot(x='keyword_ID',y='obs-exp',data=enrich[enrich.log_fdr>1.2],hue='cluster')

In [ ]:
enrich[enrich['obs-exp']<-10]

In [ ]:
c = 0
for i in enrich.sort_values(by='obs-exp').index:
    enrich.loc[i,'rank'] = c
    c = c+1
#break

In [ ]:
len(enrich[enrich.log_fdr>1.5])/len(enrich)

In [ ]:
set(enrich[enrich.log_fdr>1.5].keyword_ID)

In [ ]:
enrich[enrich.keyword_ID=='Neurodegeneration']

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))#
ax.grid()
sns.scatterplot(y='rank',x='obs-exp',data=enrich,color='gray',size='log_fdr',size_norm=norm,sizes=(10,100),edgecolor=None)
sns.scatterplot(y='rank',x='obs-exp',data=enrich[enrich.log_fdr>2],color='#125F99',size='log_fdr',size_norm=norm,sizes=(10,100),edgecolor=None,alpha=1)
plt.locator_params(axis='y',nbins=5)
# plt.xlim([-20,20])
# plt.ylim([-40,900])
texts = []
for i in enrich[enrich.log_fdr>2].index:
    texts.append(ax.text(enrich.loc[i,'obs-exp'],enrich.loc[i,'rank'],
        '%s'%enrich.loc[i,'keyword_ID'],
                      fontsize=12))
    
adjust_text(texts, autoalign='y',ax=ax,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('obs_exp_Keywords_gmm.svg',bbox_inches='tight',transparent=True)

In [ ]:
#for i in high_mdSAM_KeyWord:
sns.scatterplot(x='x',y='y',data=df_membrane)
sns.scatterplot(x='x',y='y',data=df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find('KW-0805')!=-1].Entry.values)])#0804


In [ ]:
fig, ax = plt.subplots(figsize=(3,4))#
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=20,color='#DEE0E3')
plt.legend([],frameon=False)
ax.axis('off')
for i,j in zip(list(range(3)), color1):#,'#E73490'['#674CAC','#BB1919','#3C4776']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1.0,plot_kwargs={'color':j,'lw':2})
sns.scatterplot(x='x',y='y',edgecolor=None,color='k',
                data=df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find('KW-0804')!=-1].Entry.values)])#0804
ax.set_xlim([-60,55])
ax.set_ylim([-60,55])
plt.savefig('tnse_GMM_Transcription.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find('KW-0010')!=-1].Entry.values)]

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))#
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=20,color='#DEE0E3')
plt.legend([],frameon=False)
ax.axis('off')
for i,j in zip(list(range(3)), color1):#,'#E73490'['#674CAC','#BB1919','#3C4776']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1.0,plot_kwargs={'color':j,'lw':2})
sns.scatterplot(x='x',y='y',edgecolor=None,color='k',
                data=df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find('KW-0523')!=-1].Entry.values)])#0804
ax.set_xlim([-60,55])
ax.set_ylim([-60,55])
plt.savefig('tnse_GMM_Neurodegeneration.svg',bbox_inches='tight',transparent=True)

In [ ]:
enrich[enrich.keyword_ID.str.find('receptor')!=-1]
fig, ax = plt.subplots(figsize=(4,4))#
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=20,color='#DEE0E3')
plt.legend([],frameon=False)
ax.axis('off')
for i,j in zip(list(range(10)), color1):#,'#E73490'['#674CAC','#BB1919','#3C4776']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1.0,plot_kwargs={'color':j,'lw':2})
sns.scatterplot(x='x',y='y',edgecolor=None,color='#C1312F',
                data=df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find('KW-0153')!=-1].Entry.values)])#0804
plt.savefig('tnse_GMM_0153.svg',bbox_inches='tight',transparent=True)

In [ ]:
enrich[enrich.keyword_ID.str.find('receptor')!=-1]
fig, ax = plt.subplots(figsize=(3,4))#
sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=20,color='#DEE0E3')
plt.legend([],frameon=False)
ax.axis('off')
for i,j in zip(list(range(3)), color1):#,'#E73490'['#674CAC','#BB1919','#3C4776']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1.0,plot_kwargs={'color':j,'lw':2})
sns.scatterplot(x='x',y='y',edgecolor=None,color='k',
                data=df_membrane[df_membrane.acc.isin(uniport[uniport['Keyword ID'].str.find('KW-0130')!=-1].Entry.values)])#0804
ax.set_xlim([-60,55])
ax.set_ylim([-60,55])
plt.savefig('tnse_GMM_Cell_adhesion.svg',bbox_inches='tight',transparent=True)

In [ ]:
enrich[enrich.key=='KW-0130']

In [ ]:
enrich

In [ ]:
UMAP_plot_test = pd.DataFrame()
UMAP_plot['x'] =principalDf.loc[:, 'x'] # #tsne_features[:,0] #
UMAP_plot['y'] =principalDf.loc[:, 'y'] 
principalDf

In [ ]:
UMAP_plot = pd.DataFrame()
UMAP_plot['x'] =principalDf.loc[:, 'x'] # #tsne_features[:,0] #
UMAP_plot['y'] =principalDf.loc[:, 'y'] # #tsne_features[:,0] #
#UMAP_plot_test['ENSG'] = umap_df['ENSG']
# fig, ax = plt.subplots(figsize=(3, 3))#
fig, ax = plt.subplots(figsize=(3, 3))
sns.regplot(x='x',y='y',data=UMAP_plot,fit_reg = False, color = '#C03830',marker='.',scatter_kws={'s':100,'alpha':.7,'linewidths':.3,'edgecolor':'k','rasterized':True})
#sns.regplot(x='x',y='y',data=UMAP_plot_test,fit_reg = False, color = '#C03830',marker='.',scatter_kws={'s':100,'alpha':.7,'linewidths':.3,'edgecolor':'k','rasterized':True})
# f,ax =  plt.subplots(nrows=1, ncols=1, sharey=False,figsize=(10, 10))
# sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot_test,edgecolor='k',s=20 ,ax=ax,color=".5")
ax.set_title('UMAP',fontsize= 10)
#sns.despine()
#sns.set(font_scale=2,style='whitegrid',rc={'grid.linestyle': '-','grid.linewidth': .5})
# plt.locator_params(axis="x", nbins=1)
# plt.locator_params(axis="y", nbins=1)  
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
sns.despine()
plt.xlabel('$tnse\;dim\;1$')
# plt.xticks([])
# plt.yticks([])
plt.ylabel('$tnse\;dim\;2$')
#plt.savefig('UMAP.svg',transparent=True,dpi=300)

In [ ]:
UMAP_plot = pd.DataFrame()
UMAP_plot['x'] =principalDf.loc[:, 'x'] # #tsne_features[:,0] #
UMAP_plot['y'] =principalDf.loc[:, 'y'] # #tsne_features[:,0] #
for i in UMAP_plot.index:
    UMAP_plot.loc[i,'acc']=umap_df.loc[i,'acc']
    UMAP_plot.loc[i,'AAseq']=umap_df.loc[i,'AAseq']
    UMAP_plot.loc[i,'ENST']=umap_df.loc[i,'ENST']
    UMAP_plot.loc[i,'Agg_val'] = len(umap_df.loc[i,'AAseq'])
    UMAP_plot.loc[i,'delta'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_delta()
    UMAP_plot.loc[i,'hydropathy'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_uversky_hydropathy()
    UMAP_plot.loc[i,'fraction_disorder_promoting'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_fraction_disorder_promoting()
    UMAP_plot.loc[i,'length'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_length()
    UMAP_plot.loc[i,'FCR'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_FCR()
    UMAP_plot.loc[i,'NCPR'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_NCPR()
    UMAP_plot.loc[i,'Omega'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_Omega()
    UMAP_plot.loc[i,'mean_net_charge'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_mean_net_charge()
    UMAP_plot.loc[i,'Ala'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']
    UMAP_plot.loc[i,'Cys'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C']
    UMAP_plot.loc[i,'Asp'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']
    UMAP_plot.loc[i,'Glu'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E']
    UMAP_plot.loc[i,'Phe'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F']
    UMAP_plot.loc[i,'Gly'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G']
    UMAP_plot.loc[i,'His'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']
    UMAP_plot.loc[i,'Ile'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']
    UMAP_plot.loc[i,'Lys'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K']
    UMAP_plot.loc[i,'Leu'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']
    UMAP_plot.loc[i,'Met'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']
    UMAP_plot.loc[i,'Asn'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']
    UMAP_plot.loc[i,'Pro'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['P']
    UMAP_plot.loc[i,'Gln'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']
    UMAP_plot.loc[i,'Arg'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']
    UMAP_plot.loc[i,'Ser'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']
    UMAP_plot.loc[i,'Thr'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']
    UMAP_plot.loc[i,'Val'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']
    UMAP_plot.loc[i,'Trp'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']
    UMAP_plot.loc[i,'Tyr'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']
    UMAP_plot.loc[i,'Aromatic'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F'])
    UMAP_plot.loc[i,'Posetive'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K'])
    UMAP_plot.loc[i,'Negitive'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E'])
    UMAP_plot.loc[i,'Hydro'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G'])
    UMAP_plot.loc[i,'Polar'] = 100*(SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C'])
    UMAP_plot.loc[i,'beta'] = np.mean([MUNV940103[i] for i in umap_df.loc[i,'AAseq']])
df_membrane  = UMAP_plot[~pd.isna(UMAP_plot.acc)]#
df_membrane = df_membrane.drop_duplicates(subset='AAseq')

In [ ]:
fig = plt.figure(figsize = (4,6),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=2,gridspec_kw = {'hspace':.2,'wspace':.2})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
l =['Negitive','Posetive','Aromatic','Hydro','Polar','Agg_val']
ll = ['$Negative(DE)$','$Positive(RKH)$','$Aromatic(YWF)$',
      '$Hydrophobic(GAVILM)$','$Polar(TSNQC)$','$Length$']
# ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
#       '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
#ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
ll_colors  = [matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#3A68AE"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#C03830"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#F89B35"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#924E81E8"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#6AB6B7"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#B4B5B4","#B4B5B4"])
              ,'RdGy_r','RdGy_r','RdGy_r','RdGy_r','RdGy_r']

for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[i[0]],palette=ll_colors[i[0]],
                    ax=axs.flat[i[0]],linewidth=0,edgecolor='k',alpha=1,s=50, rasterized=False)
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title(ll[i[0]],fontsize=10)
    axs.flat[i[0]].legend([],frameon=False)
    axs.flat[i[0]].axis('off')
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    axs.flat[i[0]].set_xticklabels(['','', ''])
    axs.flat[i[0]].set_yticklabels(['','', ''])
    sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]])
    sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.6,aspect=10)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    plt.yticks([0,7.5,15],('', '', ''))
    axs.flat[i[0]].set_xlabel('')
    axs.flat[i[0]].set_ylabel('')
plt.tight_layout()
#plt.savefig('all_UMAP_madan_colors_paper_not_rasterized_v3.svg',transparent=True)
#plt.savefig('all_UMAP_colors_paper_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
import csv
GPCRdb = pd.read_csv('/Users/aelazar/notebooks_for_paper/GPCR_ThT/Book1.txt',delimiter='\t',quoting=csv.QUOTE_NONE, error_bad_lines=False) # https://wiki.stjude.org/display/C3D/GPCR+common+data
transports = pd.read_csv('/Users/aelazar/Downloads/uniprotkb_Transport_AND_reviewed_true_A_2023_07_15.tsv',delimiter='\t')


In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
sns.scatterplot(x='x',y='y',data=UMAP_plot,color = '#B4B5B4',alpha=.9,edgecolor=None)
sns.scatterplot(x='x',y='y',data=df_membrane[df_membrane.acc.isin(GPCRdb['#UniProt accession'])]
                ,color = '#B55B70',alpha=.9,edgecolor=None)

plt.xticks([0,7.5,15],('', '', ''))
plt.yticks([0,7.5,15],('','', ''))
plt.xlabel('')
plt.ylabel('')
plt.xlabel('$UMAP\;dim\;1$')
# plt.xticks([])
# plt.yticks([])
plt.ylabel('$UMAP\;dim\;2$')
ax.grid(color='#B4B5B4')
plt.title('$GPCRs$')
#plt.savefig('UMAP_GPCRs.svg',transparent=True,dpi=300)

In [ ]:
df_membrane[df_membrane.acc.isin(GPCRdb['#UniProt accession'])]

In [ ]:
transports = pd.read_csv('/Users/aelazar/Downloads/uniprotkb_Transport_AND_reviewed_true_A_2023_07_15.tsv',delimiter='\t')

In [ ]:
df_membrane.to_pickle('df_membrane_tnse.pkl')

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
sns.scatterplot(x='x',y='y',data=UMAP_plot,color = '#B4B5B4',alpha=.9,edgecolor=None,s=10)
sns.scatterplot(x='x',y='y',data=df_membrane[df_membrane.acc.isin(transports.Entry)],
                color = '#6B86A2',alpha=.9,edgecolor=None,s=11)
plt.xticks([0,7.5,15],('', '', ''))
plt.yticks([0,7.5,15],('','', ''))
plt.xlabel('')
plt.ylabel('')
plt.xlabel('$UMAP\;dim\;1$')
# plt.xticks([])
# plt.yticks([])
plt.ylabel('$UMAP\;dim\;2$')
ax.grid(color='#B4B5B4')
plt.title('$Transporters$')

#plt.savefig('UMAP_transports.svg',transparent=True,dpi=300)

In [ ]:
fig = plt.figure(figsize = (6,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.2,'wspace':.2})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
sns.scatterplot(x='x',y='y',data=UMAP_plot,color = '#B4B5B4',alpha=.3,edgecolor=None,s=10,ax=axs[0])
sns.scatterplot(x='x',y='y',data=df_membrane[df_membrane.acc.isin(GPCRdb['#UniProt accession'])],s=20
                ,color = '#4388A9',alpha=1,edgecolor=None,ax=axs[0])

sns.scatterplot(x='x',y='y',data=UMAP_plot,color = '#B4B5B4',alpha=.3,edgecolor=None,s=10,ax=axs[1])
sns.scatterplot(x='x',y='y',data=df_membrane[df_membrane.acc.isin(transports.Entry)],s=20
                ,color = '#234A5D',alpha=1,edgecolor=None,ax=axs[1])
for i in range(2):
    axs[i].axis('off')
axs[0].set_title('GPCRs',fontsize=16)
axs[1].set_title('Transports',fontsize=16)
plt.savefig('all_mdSAM_families.png',transparent=True,dpi=300)

In [ ]:
MUNV940103 ={'A':1.08,'L':0.789,'R':0.976,'K':1.026,'N':1.197,'M':.812,'D':1.266,'F':0.685,
             'C':.733,'P':1.412,'Q':1.05,'S':0.987,'E':1.085,'T':0.784,'G':1.104,'W':0.755,'H':0.906,
             'Y':0.665,'I':0.583,'V':0.546}
UMAP_plot = pd.DataFrame()
UMAP_plot['x'] =principalDf.loc[:, 'x'] # #tsne_features[:,0] #
UMAP_plot['y'] =principalDf.loc[:, 'y'] # #tsne_features[:,0] #
for i in UMAP_plot.index:
    UMAP_plot.loc[i,'acc']=umap_df.loc[i,'acc']
    UMAP_plot.loc[i,'AAseq']=umap_df.loc[i,'AAseq']
    UMAP_plot.loc[i,'ENST']=umap_df.loc[i,'ENST']
    UMAP_plot.loc[i,'Agg_val'] = len(umap_df.loc[i,'AAseq'])
    UMAP_plot.loc[i,'delta'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_delta()
    UMAP_plot.loc[i,'hydropathy'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_uversky_hydropathy()
    UMAP_plot.loc[i,'fraction_disorder_promoting'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_fraction_disorder_promoting()
    UMAP_plot.loc[i,'length'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_length()
    UMAP_plot.loc[i,'FCR'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_FCR()
    UMAP_plot.loc[i,'NCPR'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_NCPR()
    UMAP_plot.loc[i,'Omega'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_Omega()
    UMAP_plot.loc[i,'mean_net_charge'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_mean_net_charge()
    UMAP_plot.loc[i,'Ala'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']
    UMAP_plot.loc[i,'Cys'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C']
    UMAP_plot.loc[i,'Asp'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']
    UMAP_plot.loc[i,'Glu'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E']
    UMAP_plot.loc[i,'Phe'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F']
    UMAP_plot.loc[i,'Gly'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G']
    UMAP_plot.loc[i,'His'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']
    UMAP_plot.loc[i,'Ile'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']
    UMAP_plot.loc[i,'Lys'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K']
    UMAP_plot.loc[i,'Leu'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']
    UMAP_plot.loc[i,'Met'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']
    UMAP_plot.loc[i,'Asn'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']
    UMAP_plot.loc[i,'Pro'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['P']
    UMAP_plot.loc[i,'Gln'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']
    UMAP_plot.loc[i,'Arg'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']
    UMAP_plot.loc[i,'Ser'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']
    UMAP_plot.loc[i,'Thr'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']
    UMAP_plot.loc[i,'Val'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']
    UMAP_plot.loc[i,'Trp'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']
    UMAP_plot.loc[i,'Tyr'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']
    UMAP_plot.loc[i,'Aromatic'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Y']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['W']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['F']
    UMAP_plot.loc[i,'Posetive'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['H']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['R']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['K']
    UMAP_plot.loc[i,'Negitive'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['D']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['E']
    UMAP_plot.loc[i,'Hydro'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['V']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['I']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['L']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['A']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['M']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['G']
    UMAP_plot.loc[i,'Polar'] = SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['T']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['S']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['N']+ SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['Q']+SequenceParameters(umap_df.loc[i,'AAseq']).get_amino_acid_fractions()['C']
    UMAP_plot.loc[i,'beta'] = np.mean([MUNV940103[i] for i in umap_df.loc[i,'AAseq']])


In [ ]:
df_membrane

In [ ]:
#!pwd
df_membrane.to_pickle('HTS/umap.pkl')

In [ ]:
import matplotlib 

In [ ]:
fig = plt.figure(figsize = (4,6),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=2,gridspec_kw = {'hspace':.2,'wspace':.2})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
l =['Negitive','Posetive','Aromatic','Hydro','Polar','Agg_val']
ll = ['$Negative(DE)$','$Positive(RKH)$','$Aromatic(YWF)$',
      '$Hydrophobic(GAVILM)$','$Polar(TSNQC)$','$Length$']
# ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
#       '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
#ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
ll_colors  = [matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#3A68AE"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#C03830"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#F89B35"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#924E81E8"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#6AB6B7"]),
              matplotlib.colors.LinearSegmentedColormap.from_list("", ["#B4B5B4","#B4B5B4"])
              ,'RdGy_r','RdGy_r','RdGy_r','RdGy_r','RdGy_r']

for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[i[0]],palette=ll_colors[i[0]],
                    ax=axs.flat[i[0]],linewidth=0,edgecolor='k',alpha=1,s=50, rasterized=False)
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title(ll[i[0]],fontsize=10)
    axs.flat[i[0]].legend([],frameon=False)
    #axs.flat[i[0]].axis('off')
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    axs.flat[i[0]].set_xticklabels(['','', ''])
    axs.flat[i[0]].set_yticklabels(['','', ''])
    sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]])
    sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.6,aspect=10)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    plt.yticks([0,7.5,15],('', '', ''))
    axs.flat[i[0]].set_xlabel('')
    axs.flat[i[0]].set_ylabel('')
plt.tight_layout()
#plt.savefig('all_UMAP_madan_colors_paper_not_rasterized_v3.svg',transparent=True)
#plt.savefig('all_UMAP_colors_paper_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
frac_plot_df = pd.read_pickle('HPA_anaysis/frac_plot_df.pkl')


In [ ]:
for ind in df_membrane.index:
    acc = df_membrane.loc[ind,'acc']
    if df_membrane.loc[ind,'acc'] in set(frac_plot_df.acc):
        print('A')
        df_membrane.loc[ind,'gmm'] = frac_plot_df[frac_plot_df.acc==acc].gmm_new.values[0]
    else:
        df_membrane.loc[ind,'gmm'] = 'No-data'#frac_plot_df[frac_plot_df.acc==acc].gmm_new.values[0]
    #frac_plot_df[frac_plot_df.acc==acc].gmm_new.values[0]#    frac_plot_df[frac_plot_df.acc==df_membrane.loc[0,'acc']]

In [ ]:
df_membrane.loc[df_membrane[df_membrane.gmm==0].index,'gmm_names'] = '$PM$'
df_membrane.loc[df_membrane[df_membrane.gmm==1].index,'gmm_names'] = '$ER$'
df_membrane.loc[df_membrane[df_membrane.gmm==2].index,'gmm_names'] = '$Endo/Lyso$'
df_membrane.loc[df_membrane[df_membrane.gmm==3].index,'gmm_names'] = '$Golgi$'
df_membrane.loc[df_membrane[df_membrane.gmm==4].index,'gmm_names'] = '$Other$'

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
# df_membrane[(df_membrane.k_means==0)].gmm.value_counts()
# sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.gmm=='No-data'],c='#D0D3D6',
#                     linewidth=0,edgecolor='k',alpha=1,s=50, rasterized=False,palette='tab10')
sns.scatterplot(x = 'x', y= 'y',data=df_membrane,color='#B4B5B4',ax =ax,linewidth=0,edgecolor=None,alpha=.9,s=20,rasterized=False)#,ax=ax)

sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.gmm!='No-data'],hue='gmm_names',palette=['#95AAD3','#23A993','#EDAEAE','#E0A458','#7E7CCB'],
                     linewidth=0,edgecolor=None,alpha=1,s=50, rasterized=False,hue_order=['$PM$','$ER$','$Endo/Lyso$','$Golgi$','$Other$'])
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(['','', ''])
ax.set_yticklabels(['','', ''])
ax.grid()
plt.legend(title='',fontsize=6,ncol=5)
plt.savefig('HPA_anaysis/UMAP_HPA.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane[df_membrane.gmm!='No-data']

In [ ]:
for i in df_membrane[df_membrane.gmm!='No-data'].index:
    if df_membrane.loc[i,'gmm'] ==1 :
        df_membrane.loc[i,'gmm_cond'] = 0
    else:
        df_membrane.loc[i,'gmm_cond'] = 1

In [ ]:
sns.scatterplot(x='x',y='y',data=df_membrane,hue='gmm_cond'
                ,edgecolor=None,s=105,alpha=1,palette='tab10')


In [ ]:
import tqdm

In [ ]:
for jj in [157]:#range(1,10001):
    for n in [10]:#range(10,15):
    #print(n)
        gmm = mixture.GaussianMixture(n_components=n, covariance_type="diag",random_state=jj)
        gmm.fit(df_membrane.loc[:,['x','y']])

        for i in enumerate(gmm.predict(df_membrane.loc[:,['x','y']])):
            df_membrane.at[df_membrane.index[i[0]],'gmm_dSAMs'] = i[1]
        kmeans = cluster.KMeans(n_clusters=n, random_state=0).fit(df_membrane.loc[:,['x','y']])
        xx = [] 
        for i in enumerate(df_membrane.index):    
            xx.append({'x':df_membrane.loc[i[1],'x'],'y':df_membrane.loc[i[1],'y'],'clus': kmeans.labels_[i[0]]})
        x3 =pd.DataFrame(xx)


        for i in enumerate(df_membrane.index):
            df_membrane.loc[i[1],'k_means']= x3.loc[i[0],'clus']
        fig = plt.figure(figsize = (10,10),constrained_layout=False)
        sns.scatterplot(x='x',y='y',data=df_membrane,hue='gmm_dSAMs',edgecolor=None,s=30,alpha=1,palette='tab20')
        sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.gmm_cond.isin([0,1])],hue='gmm_cond',palette=['k','gray'],
                        linewidth=0,edgecolor='k',alpha=1,s=30, rasterized=False)
        # #plt.title('$K-means=10$')
        #plt.legend([])
        l = []
        for i in range(n):
            for j in [0,1]:
                l.append({'k_means':i,'gmm':j,'val':len(df_membrane[(df_membrane.gmm_cond==j)&(df_membrane.gmm_dSAMs==i)])})
        table = pd.DataFrame(l).pivot_table(index='k_means',columns='gmm',values='val')
        if len(table.loc[(table==0).all(axis=1)])==0:
            res =chi2_contingency(table)
        if (res[1])<0.01:
            print(jj,n,res[1])

In [ ]:
fig = plt.figure(figsize = (15,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=5,gridspec_kw = {'hspace':.2,'wspace':.2})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
#colors = ['#936B88','#59817F','#818181','#A6558F','#3D817E']
colors = ['#936B88','#59817F','#818181','#A6558F','#3D817E']
colors=['#C6C6C8','#5A75A3','#76A5CB','#8EB6D5','#BB5450']
colors=['#C6C6C8','#808285','#221F20','#414042','#939598']
names = ['PM','ER','Endo/Lyso','Golgi','Other']
for i in range(5):
    sns.scatterplot(x='x',y='y',data=df_membrane[df_membrane.gmm==i],edgecolor='k',s=30,alpha=1,c=colors[i],zorder=2,ax=axs[i],rasterized=True)
    sns.scatterplot(x='x',y='y',data=df_membrane,edgecolor=None,s=10,alpha=1,c='#B4B5B4',zorder=1,ax=axs[i],rasterized=True)
    axs[i].set_title('%s'%names[i],fontsize=16)
for i in range(5):
    axs[i].axis('off')
plt.savefig('all_mdSAM_organelle.png',transparent=True,dpi=900)
plt.savefig('all_mdSAM_organelle.svg',transparent=True,dpi=900)

In [ ]:
df_membrane[df_membrane.gmm==0]

In [ ]:
from scipy.stats import chi2_contingency
#print(pd.DataFrame(l).pivot_table(index='k_means',columns='gmm',values='val'))

#sns.clustermap(table-res[-1],col_cluster=False,figsize=(5,5),dendrogram_ratio=0.1,)
#fig, ax = plt.subplots(figsize=(8, 8))
#fig = plt.figure(figsize = (1,10),constrained_layout=False)
ax = sns.clustermap(table-res[-1],annot=True,cmap='RdGy',fmt='.1f',figsize=(3,10))
#ax = sns.clustermap(table/res[-1],annot=True,cmap='RdGy',fmt='.1f',figsize=(4,5))


In [ ]:
res

In [ ]:
fig = plt.figure(figsize = (5,7),constrained_layout=False)
axs = fig.subplots(nrows=3, ncols=2,gridspec_kw = {'hspace':.3,'wspace':.3})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
l =['Negitive','Posetive','Aromatic','Hydro','Polar','Agg_val']
ll = ['$Negative(DE)$','$Positive(RKH)$','$Aromatic(YWF)$',
      '$Hydrophobic(GAVILM)$','$Polar(TSNQC)$','$Length$']
# ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
#       '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=df_membrane,hue=l[i[0]],palette=ll_colors[i[0]],ax=axs.flat[i[0]],linewidth=.1,edgecolor='k',alpha=.5,s=50, rasterized=True)
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title(ll[i[0]],fontsize=10)
    axs.flat[i[0]].legend([],frameon=False)
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    axs.flat[i[0]].set_xticklabels(['','', ''])
    axs.flat[i[0]].set_yticklabels(['','', ''])
    axs.flat[i[0]].axis('off')
    sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]])
    sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.6,aspect=15)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    plt.yticks([0,7.5,15],('', '', ''))
    axs.flat[i[0]].set_xlabel('')
    axs.flat[i[0]].set_ylabel('')
plt.tight_layout()
plt.savefig('all_UMAP_colors_paper.svg',transparent=True)

In [ ]:
for i in df_membrane.index:
    acc = df_membrane.loc[i,'acc']
    AAseq = df_membrane.loc[i,'AAseq']
    s = int(AF2[AF2.acc==acc].AAseq.str.find(AAseq).values[0])
    e = s + len(AAseq)
    df_membrane.loc[i,'start'] = s
    df_membrane.loc[i,'end'] = e



In [ ]:
df_membrane

In [ ]:
#df_membrane.loc[:,['x','y','acc','AAseq','start','end']].to_pickle('balint_tsv_dSAMs_umap.pkl')#.to_csv('balint_tsv_dSAMs_umap.tsv',sep='\t')

In [ ]:
pep_dict = {'SHISA8':'GGCVQVQMG','+_':'LATVQV','Desmocollin':'QQNLIVSNT','+_v2':'LVMELD','KIR2DL5B':'LDHCVFTQT','None':'none','CD44V3':'MVHLVNK','+':'VQIVYI',
            'TNFRSF13C':'DKVIILS','DDW':'none','EGF':'QHMLVAEIMV','CD22':'YSELIQFGV','TNFRSF11A':'GACEGVLLLTL','NOTCH2':'NLSVQVS','CD33':'AGVVHGA','FLT4':'GYLSIIMD','EpoR':'KYLYLVVSD','NGFR':'AGVVTTVMG','PCDHG':'SHFVGVDGV','Cadherin':'VRENIITYD','CD44V1':'GGYIFYTF',
            'TNFRSF10A':'FVVVGVLLQVV','CD44V2':'GSFGVTAVTV','FGFRL1':'VHQHIHYQC','EpoRV2':'YLVLDKWL','IFNAR1':'KCFIIENIS','APP':'YEVHHQKLVFFA','IL2RB':'ACQVYFTYD','NRG1':'HSVIVMSS'}

In [ ]:
tested_index=[]
for i in pep_dict.values():
    if df_membrane[df_membrane.AAseq.str.contains(i)].empty ==False:
        #print(df_membrane[df_membrane.AAseq.str.contains(i)].head(1))
        tested_index.append(df_membrane[df_membrane.AAseq.str.contains(i)].head(1).index.values[0])
    else:
        print(i)
        

In [ ]:
def seqlocmembrane_IDR_balant(acc,seq,verb=False):
    v =  AF2[AF2.acc==acc].AAseq.str.find(seq).values[0]
    if SSTM_membrane_boundery_v2[SSTM_membrane_boundery_v2.acc==acc].empty ==False:
        if SSTM_Cytoplasmic[SSTM_Cytoplasmic.acc==acc].empty ==False:
            log = v in SSTM_Cytoplasmic[SSTM_Cytoplasmic.acc==acc].Cytoplasmic.values[0]
            s = int(SSTM_membrane_boundery_v2[SSTM_membrane_boundery_v2.acc==acc].pos.values[0].split('-')[0])
            e = int(SSTM_membrane_boundery_v2[SSTM_membrane_boundery_v2.acc==acc].pos.values[0].split('-')[1])
            if v-s<0:
                IDR_AA = len([i for i in AF2[AF2.acc==acc].pLDDT.values[0][v:s] if i<70])
                b= balant_correction(len([i for i in AF2[AF2.acc==acc].pLDDT.values[0][v:s] if i>70]))
                total = IDR_AA+b
            else:
                IDR_AA = len([i for i in AF2[AF2.acc==acc].pLDDT.values[0][e:v] if i<70])
                b= balant_correction(len([i for i in AF2[AF2.acc==acc].pLDDT.values[0][e:v] if i>70]))
                total = IDR_AA+b
            if v in range(0,s+1):
                if log: 
                    loc = -1*total
                else:
                    loc = total
                return loc
            elif v in range(s,e):
                return np.NaN
            else:
                if log==True:   
                    loc = -1*total
                else:
                    loc = total
                return loc 
        else:
            return np.NaN
    else:
        return np.NaN

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
sns.set(font_scale=2,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
ax.grid(color='#B4B5B4')
ax = sns.scatterplot(x = 'x', y= 'y',data=df_membrane.loc[tested_index],zorder=1
                ,color='#C03830',ax =ax,linewidth=0,edgecolor='k',alpha=.7,s=100,rasterized=True)
# ax.legend([''],title_fontsize=34,fontsize=10,title="$Tested$",
#           frameon=True,fancybox=True,shadow=False,framealpha=1,loc='best')
ax.legend([''],frameon=False)
# sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.AAseq.isin(['VRENIITYD','KCFIIENIS','AGVVTTVMGS','YSELIQFGV','HSVIVMSS','QQNLIVSNT','ACQVYFTYD','YEVHHQKLVFFA',
#                                                                                     'LDHCVFTQT','GYLSIIMD','DKVIILS','QHMLVAEIMV','MVHLVNK','AGVVHGA','VHQHIHYQC','QGVIYSD','SHFVGVDGV'
#                                                                                     ,'VNSTVLVKNT'])]
#                 ,color='#C03830',ax =ax,linewidth=0,edgecolor='k',alpha=.7,s=70,rasterized=True)#,ax=ax)
ax = sns.scatterplot(x = 'x', y= 'y',data=df_membrane,color='k',
                     ax =ax,linewidth=0,edgecolor='k',alpha=.1,s=100,
                     zorder=0,rasterized=True)#,ax=ax)


#ax.set_title('$Tested$',fontsize= 24)

plt.xticks([0,7.5,15],('', '', ''))
plt.yticks([0,7.5,15],('','', ''))
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('tested_paper.svg',transparent=True)

In [ ]:
df_membrane

In [ ]:
f, axs = plt.subplots(nrows=5, ncols=4, sharey=False,figsize=(10, 10))
sns.set(font_scale=1,style='white')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Val',palette='Greys',ax =axs[0,0],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[0,0].legend(fontsize = 10,title="")
axs[0,0].set_title('V',fontsize= 16)
axs[0,0].set_xlabel('')
axs[0,0].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Ile',palette='Greys',ax=axs[0,1],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[0,1].legend(fontsize = 10,title="")
axs[0,1].set_title('I',fontsize= 16)
axs[0,1].set_xlabel('')
axs[0,1].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Leu',palette='Greys',ax=axs[0,2],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[0,2].legend(fontsize = 10,title="")
axs[0,2].set_title('L',fontsize= 16)
axs[0,2].set_xlabel('')
axs[0,2].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Met',palette='Greys',ax=axs[0,3],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[0,3].legend(fontsize = 10,title="")
axs[0,3].set_title('M',fontsize= 16)
axs[0,3].set_xlabel('')
axs[0,3].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Phe',palette='Greys',ax=axs[1,0],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[1,0].legend(fontsize = 10,title="")
axs[1,0].set_title('F',fontsize= 16)
axs[1,0].set_xlabel('')
axs[1,0].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Ser',palette='Greens',ax=axs[1,1],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[1,1].legend(fontsize = 10,title="")
axs[1,1].set_title('S',fontsize= 16)
axs[1,1].set_xlabel('')
axs[1,1].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Thr',palette='Greens',ax=axs[1,2],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[1,2].legend(fontsize = 10,title="")
axs[1,2].set_title('T',fontsize= 16)
axs[1,2].set_xlabel('')
axs[1,2].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Cys',palette='Oranges',ax=axs[1,3],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[1,3].legend(fontsize = 10,title="")
axs[1,3].set_title('C',fontsize= 16)
axs[1,3].set_xlabel('')
axs[1,3].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Gly',palette='Oranges',ax=axs[2,0],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[2,0].legend(fontsize = 10,title="")
axs[2,0].set_title('G',fontsize= 16)
axs[2,0].set_xlabel('')
axs[2,0].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Pro',palette='Greys_r',ax=axs[2,1],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[2,1].legend(fontsize = 10,title="")
axs[2,1].set_title('P',fontsize= 16)
axs[2,1].set_xlabel('')
axs[2,1].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Asn',palette='Greys',ax=axs[2,2],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[2,2].legend(fontsize = 10,title="")
axs[2,2].set_title('N',fontsize= 16)
axs[2,2].set_xlabel('')
axs[2,2].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Gln',palette='Greys',ax=axs[2,3],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[2,3].legend(fontsize = 10,title="")
axs[2,3].set_title('Q',fontsize= 16)
axs[2,3].set_xlabel('')
axs[2,3].set_ylabel('')

sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Arg',palette='Blues',ax=axs[3,0],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[3,0].legend(fontsize = 10,title="")
axs[3,0].set_title('R',fontsize= 16)
axs[3,0].set_xlabel('')
axs[3,0].set_ylabel('')

sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Lys',palette='Blues',ax=axs[3,1],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[3,1].legend(fontsize = 10,title="")
axs[3,1].set_title('K',fontsize= 16)
axs[3,1].set_xlabel('')
axs[3,1].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='His',palette='Blues',ax=axs[3,2],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[3,2].legend(fontsize = 10,title="")
axs[3,2].set_title('H',fontsize= 16)
axs[3,2].set_xlabel('')
axs[3,2].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Asp',palette='Reds',ax=axs[3,3],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[3,3].legend(fontsize = 10,title="")
axs[3,3].set_title('D',fontsize= 16)
axs[3,3].set_xlabel('')
axs[3,3].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Glu',palette='Reds',ax=axs[4,0],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[4,0].legend(fontsize = 10,title="")
axs[4,0].set_title('E',fontsize= 16)
axs[4,0].set_xlabel('')
axs[4,0].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Trp',palette='Purples',ax=axs[4,1],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[4,1].legend(fontsize = 10,title="")
axs[4,1].set_title('W',fontsize= 16)
axs[4,1].set_xlabel('')
axs[4,1].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Tyr',palette='Purples',ax=axs[4,2],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[4,2].legend(fontsize = 10,title="")
axs[4,2].set_title('Y',fontsize= 16)
axs[4,2].set_xlabel('')
axs[4,2].set_ylabel('')
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Ala',palette='Greys',ax=axs[4,3],linewidth=.4,edgecolor='k',alpha=.7,s=8)
axs[4,3].legend(fontsize = 10,title="")
axs[4,3].set_title('A',fontsize= 16)
axs[4,3].set_xlabel('')
axs[4,3].set_ylabel('')
sns.despine()
plt.tight_layout()
plt.savefig('UMAP_AA_comp_Class.svg',transparent=True)


In [ ]:
['Ala', 'Cys', 'Asp', 'Glu', 'Phe', 'Gly', 'His',
       'Ile', 'Lys', 'Leu', 'Met', 'Asn', 'Pro', 'Gln', 'Arg', 'Ser', 'Thr',
       'Val', 'Trp', 'Tyr']#UMAP_plot
df_membrane= df_membrane.append({'x':100,'y':100,'Aromatic':100,'Posetive':100,'Negitive':100,'Hydro':100,'Polar':100},ignore_index=True)

In [ ]:
fig = plt.figure(figsize = (15,10),constrained_layout=False)
axs = fig.subplots(nrows=4, ncols=5,gridspec_kw = {'hspace':.2,'wspace':.25})
l =['Ala','Cys','Asp','Glu','Phe','Gly','His','Ile','Lys','Leu','Met','Asn','Pro','Gln','Arg','Ser','Thr','Val','Trp','Tyr']
ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
      '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

#ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue=l[i[0]],ax=axs.flat[i[0]],
                    linewidth=0,edgecolor='k',alpha=.5,s=10, rasterized=False,palette=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]))
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title('$%s$'%i[1],fontsize=14)
    axs.flat[i[0]].legend([],frameon=False)
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    axs.flat[i[0]].set_xticklabels(['','', ''])
    axs.flat[i[0]].set_yticklabels(['','', ''])
    sm = plt.cm.ScalarMappable(cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]))
    sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.6,aspect=6)
    #sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]])
    #sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    #fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.4)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    plt.yticks([0,7.5,15],('','', ''))
    axs.flat[i[0]].set_xlabel('')
    axs.flat[i[0]].set_ylabel('')
    axs.flat[i[0]].axis('off')
plt.tight_layout()
plt.savefig('all_AAs_colors_paper_no_axis.png',transparent=True,dpi=300)

In [ ]:
UMAP_plot

In [ ]:
UMAP_plot = UMAP_plot[:-1]
UMAP_plot= UMAP_plot.append({'x':100,'y':100,'Ala':.7, 'Cys':.7, 'Asp':.7, 'Glu':.7, 'Phe':.7, 'Gly':.7, 'His':.7,
       'Ile':.7, 'Lys':.7, 'Leu':.7, 'Met':.7, 'Asn':.7, 'Pro':.7, 'Gln':.7, 'Arg':.7, 'Ser':.7, 'Thr':.7,
       'Val':.7, 'Trp':.7, 'Tyr':.7},ignore_index=True)
norm = mpl.colors.Normalize(vmin=0, vmax=0.7)

In [ ]:
sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue='Ala',palette=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]))

In [ ]:
three2one = {'Ala':'A', 'Cys':'C', 'Asp':'D', 'Glu':'E', 'Phe':'F', 'Gly':'G', 'His':'H',
       'Ile':'I', 'Lys':'K', 'Leu':'L', 'Met':'M', 'Asn':'N', 'Pro':'P', 'Gln':'Q', 'Arg':'R', 'Ser':'S', 'Thr':'T',
       'Val':'V', 'Trp':'W', 'Tyr':'Y'}

In [ ]:
fig = plt.figure(figsize = (20,5),constrained_layout=False)
axs = fig.subplots(nrows=2, ncols=11,gridspec_kw = {'hspace':.1,'wspace':.1})
l =['Ala','Cys','Asp','Glu','Phe','Gly','His','Ile','Lys','Leu','Met','Asn','Pro','Gln','Arg','Ser','Thr','Val','Trp','Tyr']
ll = ['$\%Negative^{DE}$','$\%Positive^{RKH}$','$\%Aromatic^{YWF}$',
      '$\%Hydrophobic^{GAVILM}$','$\%Polar^{TSNQC}$','$Length$']
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

#ll_colors  = ['Reds','Blues','RdPu','OrRd','GnBu','cividis']
for i in enumerate(l):#list(set(ThT_all[~ThT_all.name.isin(['$+$','$DDW$'])].name))
    sns.scatterplot(x = 'x', y= 'y',data=UMAP_plot,hue=i[1],ax=axs.flat[i[0]],
                    linewidth=0,edgecolor='k',alpha=.5,s=10, rasterized=True,palette=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]))
    axs.flat[i[0]].grid(color='#B4B5B4')
    axs.flat[i[0]].set_title('%s'%three2one[i[1]],fontsize=14)
    axs.flat[i[0]].legend([],frameon=False)
#     if i[0]==5:
#         axs.flat[i[0]].legend([6,8,10,12,16],title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#

#     else:
#         axs.flat[i[0]].legend(title_fontsize=0,fontsize=13,title="",frameon=True,fancybox=True,shadow=False,framealpha=.5,loc='best')#[int(i) for i in np.linspace(0,100,5)]
#     axs.flat[i[0]].locator_params(axis="x", nbins=6)
    #axs.flat[i[0]].locator_params(axis="y", nbins=6)
    axs.flat[i[0]].set_xticklabels(['','', ''])
    axs.flat[i[0]].set_yticklabels(['','', ''])
    # sm = plt.cm.ScalarMappable(cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]))
    # sm.set_array(df_membrane[l[i[0]]])
    # # axs.flat[i[0]].get_legend().remove()
    # # axs.flat[i[0]].figure.colorbar(sm)
    # fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.6,aspect=6)
    #sm = plt.cm.ScalarMappable(cmap=ll_colors[i[0]])
    #sm.set_array(df_membrane[l[i[0]]])
    # axs.flat[i[0]].get_legend().remove()
    # axs.flat[i[0]].figure.colorbar(sm)
    #fig.colorbar(sm, ax=axs.flat[i[0]],location='right', shrink=0.4)
    #axs.flat[i[0]].set_xticks([0,10,20],('', '', ''))
    plt.yticks([0,7.5,15],('','', ''))
    axs.flat[i[0]].set_xlim([-60,55])
    axs.flat[i[0]].set_ylim([-60,55])
    axs.flat[i[0]].set_xlabel('')
    axs.flat[i[0]].set_ylabel('')
    axs.flat[i[0]].axis('off')
plt.tight_layout()
sm = plt.cm.ScalarMappable(cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]),norm=norm)
fig.colorbar(sm, ax=axs.flat[20],location='right', shrink=1,aspect=6)
axs.flat[20].axis('off')
axs.flat[21].axis('off')
plt.savefig('all_AAs_colors_paper_no_axis_norm_100.png',transparent=True,dpi=600)
plt.savefig('all_AAs_colors_paper_no_axis_norm_100.svg',transparent=True,dpi=600)



In [ ]:
i[0]

In [ ]:
fig, ax = plt.subplots(figsize=(2.3,2.3)) 
sm = plt.cm.ScalarMappable(cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6","#39477A"]),norm=norm)
fig.colorbar(sm, ax=ax,location='right', shrink=1,aspect=6)
plt.savefig('all_AAs_colors_paper_no_axis_norm_100_colorbar.svg',transparent=True,dpi=600)


In [ ]:
import sklearn.cluster as cluster
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph

In [ ]:
X = StandardScaler().fit_transform(df_membrane.loc[:,['x','y']])

# estimate bandwidth for mean shift
bandwidth = cluster.estimate_bandwidth(X, quantile=0.3)

# connectivity matrix for structured Ward
connectivity = kneighbors_graph(
    X, n_neighbors=10, include_self=False
)
# make connectivity symmetric
connectivity = 0.5 * (connectivity + connectivity.T)

In [ ]:
cluster.KMeans

In [ ]:
cluster

In [ ]:
kmeans = cluster.KMeans(n_clusters=9, random_state=0).fit(df_membrane.loc[:,['x','y']])
#kmeans = cluster.spectral(n_clusters=20, random_state=0).fit(df_membrane.loc[:,['x','y']])

In [ ]:

plt.legend([])
sm = plt.cm.ScalarMappable(cmap='magma')
sm.set_array(pd.DataFrame(xx))
# axs.flat[i[0]].get_legend().remove()
# axs.flat[i[0]].figure.colorbar(sm)
fig.colorbar(sm,location='right', shrink=0.4)

In [ ]:
pd.DataFrame(xx).to_pickle('HTS/kmeans.pkl')

In [ ]:
x3 =pd.DataFrame(xx)#.clus.value_counts()

In [ ]:
SSTM_membrane_boundery_v2 = pd.read_pickle('SSTM_membrane_boundery_v2.pkl')
SSTM_Cytoplasmic= pd.read_pickle('SSTM_Cytoplasmic_acc.pkl')
def balant_correction(n):
    return 0.52*((70*n)**(1/3))

In [ ]:
SSTM_Cytoplasmic

In [ ]:
for i in df_membrane[df_membrane.acc.isin(SSTM_Cytoplasmic.acc)].index:
    s = int(df_membrane.loc[i,'start'])
    e = int(df_membrane.loc[i,'end'])
    df_membrane.loc[i,'seqlocmembrane'] = seqlocmembrane_IDR_balant(df_membrane.loc[i,'acc'],AF2[AF2.acc==df_membrane.loc[i,'acc']].AAseq.values[0][s:e])

In [ ]:
len(set(df_membrane.loc[:,['x','y','acc','AAseq','seqlocmembrane','k_means']].k_means))

In [ ]:
df_membrane.loc[:,['x','y','acc','AAseq','seqlocmembrane','k_means']].to_csv('balint_tsv_dSAMs_umap_with_kmeans.tsv',sep='\t')#.to_pickle('balint_tsv_dSAMs_umap.pkl')#.to_csv('balint_tsv_dSAMs_umap.tsv',sep='\t')

In [ ]:
sns.set(font_scale=1.5,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

fig, ax = plt.subplots(figsize=(7,7)) 
# df_membrane[(df_membrane.k_means==0)].gmm.value_counts()
# sns.scatterplot(x = 'x', y= 'y',data=df_membrane[df_membrane.gmm=='No-data'],c='#D0D3D6',
#                     linewidth=0,edgecolor='k',alpha=1,s=50, rasterized=False,palette='tab10')
#sns.scatterplot(x = 'x', y= 'y',data=df_membrane,color='#B4B5B4',ax =ax,linewidth=0,edgecolor=None,alpha=.9,s=20,rasterized=False)#,ax=ax)

# sns.scatterplot(x = 'x', y= 'y',data=pd.read_csv('balint_tsv_dSAMs_umap_with_kmeans.tsv',delimiter='\t'),hue='k_means',palette=['#95AAD3','#23A993',
#                                                                                                                                 '#EDAEAE','#E0A458',
#                                                                                                                                  '#7E7CCB','#F47E3E','#7EFF70','#33FFC9',
#                                                                                                                                 '#C455A8','#096771'],
#                      linewidth=0,edgecolor=None,alpha=1,s=20, rasterized=False,)
sns.scatterplot(x = 'x', y= 'y',data=pd.read_csv('balint_tsv_dSAMs_umap_with_kmeans.tsv',delimiter='\t'),hue='k_means',palette='tab10',
                     linewidth=0,edgecolor=None,alpha=1,s=20, rasterized=False,)

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(['','', ''])
ax.set_yticklabels(['','', ''])
ax.grid()
ax.axis('off')
h, l = ax.get_legend_handles_labels()
ax.legend(h, [i.split('.')[0] for i in l])
#plt.legend(title='',fontsize=10,ncol=2)
plt.savefig('HPA_anaysis/UMAP_KMEANS.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane.loc[:,['x','y','acc','AAseq','k_means']]

In [ ]:
motif_precent = pd.read_csv('k_means_motif.tsv',delimiter='\t',index_col='motif')

In [ ]:
motif_precent.iloc[4:].transpose()

In [ ]:
sns.set(font_scale=.75,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

fig, ax = plt.subplots(figsize=(3,6)) 
sns.heatmap(motif_precent.iloc[4:],square=False,cmap='cividis',cbar_kws={"shrink": .4,'location':'right'},vmin=0)
plt.ylabel('$Regular\;expression$')
plt.xlabel('$k‑means\;cluster$')
#plt.savefig('HPA_anaysis/UMAP_KMEANS.svg',bbox_inches='tight',transparent=True)

In [ ]:
motif_precent.iloc[4:].transpose()

In [ ]:
sns.set(font_scale=.75,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

fig, ax = plt.subplots(figsize=(3,6)) 
sns.heatmap(motif_precent.iloc[0:4].transpose(),square=True,cmap='cividis',cbar_kws={"shrink": .2,'location':'right'},vmin=0,vmax=100)
plt.ylabel('$Regular\;expression$')
plt.xlabel('$k‑means\;cluster$')

In [ ]:
sns.heatmap(motif_precent.iloc[4:].transpose(),square=True,cmap='cividis',cbar_kws={"shrink": .4,'location':'right'},vmin=0)


In [ ]:
import matplotlib.gridspec as gridspec

In [ ]:
fig = plt.subplots(figsize=(25,4))
from matplotlib import colors
divnorm=colors.TwoSlopeNorm(vmin=-3, vcenter=0., vmax=.5)
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E","#B34F91"])#"#F3F3F3"
gs = gridspec.GridSpec(12, 12,figure=fig)
gs.update(left=0.05, right=0.48, wspace=0.0,hspace=0)
ax1 = plt.subplot(gs[:,0:9])
ax2 = plt.subplot(gs[:,10:])
sns.heatmap(motif_precent.iloc[4:].transpose(),square=False,cmap='RdGy_r',cbar_kws={"shrink": .4,'location':'right'},vmin=0,ax=ax1,lw=0)
sns.heatmap(motif_precent.iloc[0:4].transpose(),square=False,cmap='Greys',cbar_kws={"shrink": .4,'location':'right'},vmin=0,vmax=100,ax=ax2,lw=.5)
ax1.set_xlabel('Regular expression')
ax1.set_ylabel('K-means cluster')
plt.savefig('HPA_anaysis/KMEANS_motif.svg',bbox_inches='tight',transparent=True)

In [ ]:
plt.pie([100-73.5,73.5],labels=['$73.5\%$','$Any\;SLiM$'],colors=['#18827E','#B34F91'])
plt.savefig('HPA_anaysis/KMEANS_motif_pie.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_membrane.loc[:,['x','y','acc','AAseq','seqlocmembrane','k_means']]

In [ ]:
def AAseq2tri(AAseq):
    Ps = ['R','K','H']
    Ns = ['D','E']
    Hs = list(set.difference(set(list('ARNDCEQGHILKMFPSTWYV')),set.union(set(Ps),set(Ns))))
#['A','V','I','L','F','M','C','W','Y']
    #Hs = ['A','V','I','L','F','M','C','W','Y']
    #print(AAseq)
    Ps_sum = np.sum([AAseq.count(i) for i in Ps])
    Ns_sums = np.sum([AAseq.count(i) for i in Ns])
    Hs_sums = np.sum([AAseq.count(i) for i in Hs])
    x = tuple([Ns_sums,Ps_sum,Hs_sums]/np.sum([Ns_sums,Ps_sum,Hs_sums]))
    return (100*x[0],100*x[1],100*x[2])
    

In [ ]:
AAseq2tri(i)

In [ ]:
l = []
for i in mdSAMs.AAseq:
    pos,neg,hydro = AAseq2tri(i)
    l.append(AAseq2tri(i))
    #l.append({'AAseq':i,'pos':pos,'neg':neg,'hydro':hydro})



In [ ]:
pd.DataFrame(l).to_pickle('ternary_df.pkl')

In [ ]:
import pandas as pd 

In [ ]:
pd.read_pickle('ternary_df.pkl')

In [ ]:
x = 100*(AAseq2tri('GCTTLLS')[0]),100*(AAseq2tri('GCTTLLS')[1]),100*(AAseq2tri('GCTTLLS')[2])
type(x)

In [ ]:
seqs = ['GCTTLLS','HSVIVMSS','VLSCGCVSA','NLSVQVS','FVVVGVLLQVV','QQNLIVSNT','SAVISTTT','SGVIGLV','VSVWVLQS','AGVVHGA','CNLVYLIPH','AGVVTTVMGS','SIHIVQS','VLCQYCGLLQI','GGCVQVQMG']
#seqs  =['GCTTLLS','HSVIVMSS','VRENIITYD','RVMVYRG','VLSCGCVSA','GACEGVLLLTL','DMVEVIYINRK','SHFVGVDGV','NLSVQVS','LDHCVFTQT','FVVVGVLLQVV','RVCFWRL','ACQVYFTYD','QQNLIVSNT','KYQVRWKIIESY','KCFIIENIS','YEVHHQKLVFFA','AYIGVTYK','QLFVNLK','ITSRIFQWK','SAVISTTT','YSELIQFGV','MVHLVNK','QTLTFCWE','VNATYVNVKCV','SGVIGLV','QHMLVAEIMV','VSVVYSE','DKVIILSP','GSVDIVAMD','KYLYLVVSD','VSVWVLQS','VHQHIHYQC','TEVIIIEVD','AGVVHGA','CNLVYLIPH','LSVILDYD','GYLSIIMD','QGVIYSD','AGVVTTVMGS','SWVEFIE','SIHIVQS','VLCQYCGLLQI','GGCVQVQMG']
seqs2 = ['RVMVYRG','RVCFWRL','KYQVRWKIIESY','YEVHHQKLVFFA','AYIGVTYK','QLFVNLK','ITSRIFQWK','MVHLVNK','VNATYVNVKCV','VHQHIHYQC']
seq3 = ['VRENIITYD','GACEGVLLLTL','DMVEVIYINRK','SHFVGVDGV','LDHCVFTQT','ACQVYFTYD','KCFIIENIS','YSELIQFGV','QTLTFCWE','QHMLVAEIMV','VSVVYSE','DKVIILSP','GSVDIVAMD','KYLYLVVSD','TEVIIIEVD','LSVILDYD','GYLSIIMD','QGVIYSD','SWVEFIE']

In [ ]:
ll= []
agg = []
for i in seqs:
    ll.append(AAseq2tri(i))
    agg.append({'AAseq':i,'pos':AAseq2tri(i)[1],'neg':AAseq2tri(i)[0],'hydro':AAseq2tri(i)[2]})
l3= []
for i in seqs2:
    
    l3.append(AAseq2tri(i))
    agg.append({'AAseq':i,'pos':AAseq2tri(i)[1],'neg':AAseq2tri(i)[0],'hydro':AAseq2tri(i)[2]})

l4= []
for i in seq3:
    
    l4.append(AAseq2tri(i))
    agg.append({'AAseq':i,'pos':AAseq2tri(i)[1],'neg':AAseq2tri(i)[0],'hydro':AAseq2tri(i)[2]})


In [ ]:
sns.heatmap(pd.DataFrame(l).transpose())

In [ ]:
ThT_tested = pd.DataFrame(agg)
ThT_tested.loc[ThT_tested[ThT_tested.hydro==100].index,'cluster']=0
ThT_tested.loc[ThT_tested[(ThT_tested.neg.between(1,50))&(ThT_tested.hydro<100)&(ThT_tested.pos<=20)].index,'cluster']=1
ThT_tested.loc[ThT_tested[(ThT_tested.pos.between(1,50))&(ThT_tested.hydro<100)&(ThT_tested.neg<=10)].index,'cluster']=2
ThT_tested.to_pickle('ternary_ThT_tested.pkl')

In [ ]:
sns.clustermap(pd.DataFrame(l).transpose(),figsize=(3,3),col_cluster=1,row_cluster=0,cmap='RdGy_r',method='average')

In [ ]:
ThT_tested[ThT_tested.cluster==2]

In [ ]:
ThT_tested

In [ ]:
import ternary

fig, ax = plt.subplots(figsize=(5,5)) 
scale = 100
figure, tax = ternary.figure(scale=scale,ax=ax)
tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.cluster==0].index,['neg','pos','hydro']].values, color='g',alpha=1,s=40)
tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.cluster==1].index,['neg','pos','hydro']].values, color='b',alpha=1,s=40)
tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.cluster==2].index,['neg','pos','hydro']].values, color='r',alpha=1,s=40)
plt.axvline(10)
#tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.AAseq=='KYQVRWKIIESY'].index,['neg','pos','hydro']].values, color='orange',alpha=1,s=40)

In [ ]:
import ternary
fig, ax = plt.subplots(figsize=(5,5)) 
scale = 100
figure, tax = ternary.figure(scale=scale,ax=ax)
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
#tax.scatter(l, color='gray',alpha=.3,s=40)
tax.scatter(ll, color='#926189',s=100,alpha=.5)
# tax.scatter(l3, color='#B2493E',s=100,alpha=.5)
# tax.scatter(l4, color='#4C6CAB',s=100,alpha=.5)
# tax.scatter([(1,0,0)] color='b',alpha=1,s=100)
# tax.scatter([(0,1,0)], color='red',alpha=1,s=100)
# tax.scatter([(0,0,1)], color='orange',alpha=1,s=100)
#tax.scatter([(30,60,10)], color='orange',alpha=1,s=100)
tax.ticks(axis='lbr', linewidth=0, multiple=10,fontsize=16,offset=0.02)
tax.left_axis_label("Hydrophobic", fontsize=20,offset=0.15)
tax.right_axis_label("Positive", fontsize=20,offset=0.15)
tax.bottom_axis_label("Negative", fontsize=20,offset=0.15)
plt.axis('off')
tax.show()


In [ ]:

fig = plt.figure(figsize = (16,3),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=4,gridspec_kw = {'hspace':.3,'wspace':.3})
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
scale = 100
figure, tax = ternary.figure(scale=scale,ax=axs[0])
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
tax.scatter(l, color='k',alpha=.1,s=20,zorder=3)
tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=14,offset=0.03)
axs[0].axis('off')
axs[0].set_title('All mdSAMs',pad=30,fontsize=20)
#tax.ticks(axis='lbr', linewidth=0, multiple=10,fontsize=16,offset=0.02)
tax.left_axis_label("%Other", fontsize=16,offset=0.20)
tax.right_axis_label("%RKH", fontsize=16,offset=0.20)
tax.bottom_axis_label("%DE", fontsize=16,offset=0.20)

figure, tax = ternary.figure(scale=scale,ax=axs[1])
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.cluster==0].index,['neg','pos','hydro']].values, color='#926189',s=100,zorder=3)
tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=14,offset=0.03)
axs[1].axis('off')
tax.left_axis_label("%Other", fontsize=16,offset=0.20)
tax.right_axis_label("%RKH", fontsize=16,offset=0.20)
tax.bottom_axis_label("%DE", fontsize=16,offset=0.20)

figure, tax = ternary.figure(scale=scale,ax=axs[2])
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.cluster==2].index,['neg','pos','hydro']].values, color='#B2493E',s=100,zorder=3)
tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=14,offset=0.03)
axs[2].axis('off')
axs[2].set_title('Archetype',pad=30,fontsize=20)
tax.left_axis_label("%Other", fontsize=16,offset=0.20)
tax.right_axis_label("%RKH", fontsize=16,offset=0.20)
tax.bottom_axis_label("%DE", fontsize=16,offset=0.20)
figure, tax = ternary.figure(scale=scale,ax=axs[3])
tax.boundary(linewidth=2.0,zorder=0)
tax.gridlines(multiple=10, color="gray")
tax.scatter(ThT_tested.loc[ThT_tested[ThT_tested.cluster==1].index,['neg','pos','hydro']].values, color='#4C6CAB',s=100,zorder=3)
tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=14,offset=0.03)
axs[3].axis('off')
tax.left_axis_label("%Other", fontsize=16,offset=0.20)
tax.right_axis_label("%RKH", fontsize=16,offset=0.20)
tax.bottom_axis_label("%DE", fontsize=16,offset=0.20)

# figure, tax = ternary.figure(scale=scale,ax=axs[4])
# tax.boundary(linewidth=2.0,zorder=0)
# tax.gridlines(multiple=10, color="gray")
# tax.scatter(l4, color='#4C6CAB',s=100)
# tax.ticks(axis='lbr', linewidth=0, multiple=20,fontsize=14,offset=0.02)
# axs[4].axis('off')
# tax.left_axis_label("Hydrophobic", fontsize=20,offset=0.17)
# tax.right_axis_label("Positive", fontsize=20,offset=0.17)
# tax.bottom_axis_label("Negative", fontsize=20,offset=0.17)

tax.savefig('Archetype_ternary.svg',bbox_inches='tight',transparent=True)

In [ ]:
mdSAMs = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')

In [ ]:
from Levenshtein import distance as levenshtein_distance
lev_dist = []
for i  in tqdm(mdSAMs.index):
    for j in mdSAMs.index:
        if i!=j:
            lev_dist.append({'AAseq1':mdSAMs.loc[i,'AAseq'],'AAseq1_len':len(mdSAMs.loc[i,'AAseq']),'AAseq2_len':len(mdSAMs.loc[j,'AAseq'])
                             ,'AAseq2':mdSAMs.loc[j,'AAseq'],'lev':levenshtein_distance(mdSAMs.loc[i,'AAseq'],mdSAMs.loc[j,'AAseq'])})

In [ ]:
len(mdSAMs.index)

In [ ]:
lev_dist_df = pd.DataFrame(lev_dist)

In [ ]:
mdSAMs[mdSAMs.AA_length==7].sample(5).AAseq.values

In [ ]:
z = []
for i in range(7,16):
#    e = len(lev_dist_df[(lev_dist_df.AAseq1_len==i)|((lev_dist_df.AAseq2_len==i))])
    for j in lev_dist_df[(lev_dist_df.AAseq1_len==i)|((lev_dist_df.AAseq2_len==i))].lev:
        z.append({'length':i,'lev_dist':j})

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
ax.grid()
sns.ecdfplot(lev_dist_df.lev,stat='count',log_scale=(False,True),lw=3,color='#C03830')
plt.xlabel('Levenshtein distance',fontsize=14)
plt.ylim([10,2e6])
plt.ylabel('Count',fontsize=14)
plt.locator_params(axis='x', nbins=5)
#plt.locator_params(axis='y', nbins=5)
plt.title('All Vs All',fontsize=14)
plt.savefig('LD_all_v_all_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
zz = pd.DataFrame(z)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
ax.grid()
sns.histplot(lev_dist_df.lev,bins=10,lw=0,kde=0,color='gray',zorder=2,alpha=1,log_scale=(0,0))
plt.yscale('log')
#plt.locator_params(axis='y',nbins=5)
#ax.ticklabel_format(axis='y', style='scientific',scilimits=(0,0))#,stat='probability')
plt.xlabel('',fontsize=14)

In [ ]:
his = []
for i in range(0,18):
    his.append({'lev':i,'val':np.log10(len(lev_dist_df[lev_dist_df.lev==i]['lev']))})

In [ ]:
from mycolorpy import colorlist as mcp
color1=mcp.gen_color(cmap="PuOr",n=18)
print(color1)

In [ ]:
pd.DataFrame(his)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
sns.barplot(x='lev',y='val',data=pd.DataFrame(his),lw=1,width=1,palette=color1,edgecolor='k',ax=ax,zorder=2)
ax.grid(zorder=0)
plt.setp(ax.patches, linewidth=.0)
ax.locator_params(axis='y',nbins=4)

# ax.locator_params(axis='x',nbins=3)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
plt.savefig('LD_all_v_all_v4.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.boxenplot(lev_dist_df.lev,color='#C03830',saturation=1,zorder=2)
plt.locator_params(axis='y',nbins=5)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
plt.savefig('LD_all_v_all_v3.svg',bbox_inches='tight',transparent=True)


In [ ]:
pd.DataFrame(his).hist()

In [ ]:
from matplotlib.collections import LineCollection
fig, ax = plt.subplots(figsize=(3,3))
ax.grid(zorder=0)

ax = sns.lineplot(x='lev',y='val',data=pd.DataFrame(his),zorder=3,lw=3,color='#9E9E9E',)
#ax = sns.barplot(x='lev',y='val',data=pd.DataFrame(his),zorder=4,color='#C6C6C8',width=.75,alpha=1)
#ax = sns.scatterplot(x='lev',y='val',data=pd.DataFrame(his),zorder=4,hue='lev',palette=color1,edgecolor='k',s=50)


# x, y = ax.get_lines()[0].get_data()
# segments = np.array([x[:-1], y[:-1], x[1:], y[1:]]).T.reshape(-1, 2, 2)
# norm = plt.Normalize(x.min(), x.max())
# lc = LineCollection(segments, cmap='PuOr', norm=norm,zorder=3)
# lc.set_array(x[:-1])
# lc.set_linewidth(6)
# ax.get_lines()[0].remove()
# line = ax.add_collection(lc)
#sns.lineplot(x='lev',y='val',data=pd.DataFrame(his),color='gray',lw=5,zorder=0)
#ax.fill_between(x, y, color='#C6C6C8', alpha=.8)
ax.margins(x=0)
ax.set_ylim(ymin=0)
ax.locator_params(axis='y',nbins=4)
ax.locator_params(axis='x',nbins=6)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=12,rotation=0)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=12)
plt.xlabel('')
plt.ylabel('')
# plt.xticks([])
ax.axvline(lev_dist_df.lev.median())
plt.legend([],frameon=False)
plt.xlim([-1,18])
plt.savefig('LD_all_v_all_v5.svg',bbox_inches='tight',transparent=True)


In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
ax.grid()
sns.lineplot(x='lev',y='val',data=pd.DataFrame(his),zorder=3,lw=3,color='#9E9E9E')
ax.locator_params(axis='y',nbins=4)
ax.locator_params(axis='x',nbins=4)
ax.axvline(lev_dist_df.lev.median(),color='r',ls='--')
plt.savefig('LD_all_v_all_v5.svg',bbox_inches='tight',transparent=True)


In [ ]:
plt.stem(pd.DataFrame(his)['val'])

In [ ]:
ax = sns.lineplot(x='lev',y='val',data=pd.DataFrame(his),zorder=3,palette=color1)


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(lev_dist_df.pivot_table(values='lev',
    index='AAseq1',
    columns='AAseq2'),mask=np.triu(lev_dist_df.pivot_table(values='lev',
    index='AAseq1',
    columns='AAseq2').to_numpy()),cmap='PuOr',rasterized=True,cbar_kws={"shrink": 0.5,"ticks":[0,8,16]})
plt.yticks([])
plt.xticks([])
plt.xlabel('')
plt.ylabel('')
plt.savefig('temp_heat_lev.svg',bbox_inches='tight',transparent=True)


In [ ]:
g = sns.clustermap(l3,cmap='PuOr',rasterized=True,figsize=(6,6))

In [ ]:
cmap = matplotlib.colors.LinearSegmentedColormap.from_list('aa',["#C6C6C8","#AE5A54"])


In [ ]:
l3 = lev_dist_df.pivot_table(values='lev',
    index='AAseq1',
    columns='AAseq2')
l3 = l3.fillna(0)
mask = np.triu(np.ones_like(l3))
g = sns.clustermap(l3,cmap=cmap,rasterized=True,figsize=(6,6))
values = g.ax_heatmap.collections[0].get_array().reshape(l3.shape)
new_values = np.ma.array(values, mask=mask)
g.ax_heatmap.collections[0].set_array(new_values)
g.ax_heatmap.set_xticklabels([])
g.ax_heatmap.set_yticklabels([])
plt.savefig('temp_heat_lev_v2.svg',bbox_inches='tight',transparent=True)



In [ ]:
d = lev_dist_df.pivot_table(values='lev',
    index='AAseq1',
    columns='AAseq2')

In [ ]:
d.columns